<a href="https://colab.research.google.com/github/ohki-yu0225/toukei_I/blob/main/261120_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## シンプソンのパラドックス：相関と因果関係
運動時間と病気の発生率の擬似データ（参考：https://www.saishunkansys.com/blog/marketing/simpson_paradox/ ）。

In [ ]:
!pip install japanize_matplotlib

import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib

plt.rcParams["figure.figsize"] = (4, 3)
%config InlineBackend.figure_format = "retina"

In [ ]:
rng = np.random.default_rng(0)
n = 200

age = rng.integers(20, 81, n)  # 整数の年齢
group = np.where(age >= 50, "50歳以上", "50歳未満")
exercise = np.clip(2 + 3 * (age >= 50) + rng.normal(0, 1, n), 0, None)
disease = 10 + 0.15 * age - 1.0 * exercise + rng.normal(0, 2, n)

df = pd.DataFrame({"年齢": age, "年齢グループ": group,
                   "運動時間": exercise, "病気の発生率": disease})
df.head().round(1)

### 全体の散布図と相関

In [ ]:
r_all = df["運動時間"].corr(df["病気の発生率"])
print(f"全体の相関係数: r = {r_all:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
sns.regplot(data=df, x="運動時間", y="病気の発生率", ax=ax, ci=None,
            scatter_kws={"color": "gray", "alpha": 0.6}, line_kws={"color": "red"})
ax.set_title(f"運動時間と病気の発生率（r = {r_all:.2f}）")
ax.set_xlabel("運動時間（時間/週）")
ax.set_ylabel("病気の発生率（%）")
plt.tight_layout()
plt.show()

### 年齢で色分けした散布図

In [ ]:
for g, d in df.groupby("年齢グループ"):
    print(f"{g} の相関係数: r = {d['運動時間'].corr(d['病気の発生率']):.3f}")

palette = {"50歳未満": "tab:blue", "50歳以上": "orange"}
fig, ax = plt.subplots(figsize=(6, 5))
for g, d in df.groupby("年齢グループ"):
    r = d["運動時間"].corr(d["病気の発生率"])
    sns.regplot(data=d, x="運動時間", y="病気の発生率", ax=ax, ci=None,
                color=palette[g], label=f"{g}（r = {r:.2f}）",
                scatter_kws={"alpha": 0.7})
# 全体の回帰直線（比較用）
sns.regplot(data=df, x="運動時間", y="病気の発生率", ax=ax, ci=None, scatter=False,
            line_kws={"color": "red", "linestyle": "--"}, label=f"全体（r = {r_all:.2f}）")
ax.set_title("年齢グループで色分け")
ax.set_xlabel("運動時間（時間/週）")
ax.set_ylabel("病気の発生率（%）")
ax.legend()
plt.tight_layout()
plt.show()